# 03 – Visualizaciones finales (Acto 1)

Genera las visualizaciones publicables del Acto 1: la brecha de crecimiento entre régimen especial y definitivo.
Cada gráfico se exporta en 6 versiones via `src/utils.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Agrega src/ al path para importar utils
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from utils import exportar_grafico

# Carga datos
enc = pd.read_csv(ROOT / 'data/processed/bccr-pib-encadenados-limpio.csv')
enc = enc.query('año >= 2000').copy()

# Cálculos base
enc['pct_especial']    = enc['regimen_especial']    / enc['pib_total'] * 100
enc['pct_definitivo']  = enc['regimen_definitivo']  / enc['pib_total'] * 100

base = enc.loc[enc['año'] == 2000].iloc[0]
enc['idx_total']      = enc['pib_total']           / base['pib_total']           * 100
enc['idx_definitivo'] = enc['regimen_definitivo']  / base['regimen_definitivo']  * 100
enc['idx_especial']   = enc['regimen_especial']    / base['regimen_especial']    * 100

print('Datos cargados:', len(enc), 'años (', enc['año'].min(), '-', enc['año'].max(), ')')

## Gráfico A: Participación del régimen especial en el PIB

Una sola línea. Muestra cómo la zona franca pasó de ~5.5% a ~15% del PIB encadenado.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(enc['año'], enc['pct_especial'],
        color='#60ff12', linewidth=2.5, marker='o', markersize=4)

# Anotaciones clave
ax.annotate('5.5%\n(2000)',
            xy=(2000, enc.loc[enc['año']==2000, 'pct_especial'].values[0]),
            xytext=(2002, 4.5), fontsize=9, color='#60ff12',
            arrowprops=dict(arrowstyle='->', color='#60ff12', lw=1))

ax.annotate('~15%\n(2025)',
            xy=(2025, enc.loc[enc['año']==2025, 'pct_especial'].values[0]),
            xytext=(2021, 13), fontsize=9, color='#60ff12',
            arrowprops=dict(arrowstyle='->', color='#60ff12', lw=1))

ax.set_title('Régimen especial: participación en el PIB real\n(colones encadenados, base 2017)',
             fontsize=13, pad=14)
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('% del PIB', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_xlim(1999, 2027)
ax.grid(True, alpha=0.3)
fig.tight_layout()

exportar_grafico(fig, 'pib_pct_especial', carpeta=ROOT / 'outputs/graficos')
plt.show()

## Gráfico B: Índice de crecimiento base 2000

Tres líneas diferenciadas por color. El régimen especial crece 7x mientras el definitivo apenas 2.5x.

In [ ]:
# Paleta diferenciada: verde principal + dos neutros
COLOR_ESPECIAL   = '#60ff12'  # verde MásOpciones
COLOR_TOTAL      = '#FFFFFF'  # blanco (visible en dark)
COLOR_DEFINITIVO = '#888888'  # gris medio

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(enc['año'], enc['idx_especial'],
        color=COLOR_ESPECIAL,   linewidth=2.5, label='Régimen especial (zona franca)')
ax.plot(enc['año'], enc['idx_total'],
        color=COLOR_TOTAL,      linewidth=1.8, linestyle='--', label='PIB total')
ax.plot(enc['año'], enc['idx_definitivo'],
        color=COLOR_DEFINITIVO, linewidth=1.8, linestyle=':',  label='Régimen definitivo')

# Línea de referencia base 100
ax.axhline(100, color='#555555', linewidth=0.8, linestyle='-')

# Anotación final
ultimo = enc['año'].max()
ax.annotate(f"+{enc.loc[enc['año']==ultimo,'idx_especial'].values[0]:.0f}",
            xy=(ultimo, enc.loc[enc['año']==ultimo,'idx_especial'].values[0]),
            xytext=(-30, 10), textcoords='offset points',
            fontsize=9, color=COLOR_ESPECIAL)

ax.set_title('Crecimiento acumulado desde 2000\n(índice base 100 = año 2000, volumen encadenado)',
             fontsize=13, pad=14)
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('Índice (100 = año 2000)', fontsize=11)
ax.set_xlim(1999, 2027)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()

# Nota: exportar_grafico sobreescribe colores de línea con #60ff12 en ambos temas.
# Para este gráfico multi-línea guardamos manualmente dark y light.

outdir = ROOT / 'outputs/graficos'
outdir.mkdir(parents=True, exist_ok=True)

# --- Dark ---
fig.set_facecolor('#0D0D0D')
ax.set_facecolor('#0D0D0D')
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_color('#FFFFFF')
ax.tick_params(colors='#FFFFFF')
for spine in ax.spines.values(): spine.set_color('#FFFFFF')
leg = ax.get_legend()
if leg:
    leg.get_frame().set_facecolor('#0D0D0D')
    leg.get_frame().set_edgecolor('#3A3A3A')
    for t in leg.get_texts(): t.set_color('#FFFFFF')

for fmt, (w, h) in [('square',(1080,1080)), ('16x9',(1920,1080)), ('linkedin',(1200,627))]:
    fig.set_size_inches(w/150, h/150)
    fig.savefig(outdir / f'pib_indice_crecimiento_dark_{fmt}.png',
                dpi=150, facecolor='#0D0D0D')
    print(f'Exportado: pib_indice_crecimiento_dark_{fmt}.png')

# --- Light ---
fig.set_facecolor('#F5F5F5')
ax.set_facecolor('#F5F5F5')
for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
             ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_color('#000000')
ax.tick_params(colors='#000000')
for spine in ax.spines.values(): spine.set_color('#000000')
if leg:
    leg.get_frame().set_facecolor('#F5F5F5')
    leg.get_frame().set_edgecolor('#D9D9D9')
    for t in leg.get_texts(): t.set_color('#000000')

for fmt, (w, h) in [('square',(1080,1080)), ('16x9',(1920,1080)), ('linkedin',(1200,627))]:
    fig.set_size_inches(w/150, h/150)
    fig.savefig(outdir / f'pib_indice_crecimiento_light_{fmt}.png',
                dpi=150, facecolor='#F5F5F5')
    print(f'Exportado: pib_indice_crecimiento_light_{fmt}.png')

plt.show()